In [ ]:
import numpy as np
import pandas as pd
import json
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from transformers import pipeline
from sklearn.metrics import accuracy_score,f1_score


In [ ]:
with open('/content/drive/MyDrive/DeepLearning/zero-shot dynamic routing system/data_full.json', 'r') as file:
    data = json.load(file)

In [ ]:
df_train = pd.DataFrame(data['train'], columns=['utterance', 'intent'])
df_test = pd.DataFrame(data['test'], columns=['utterance', 'intent'])

In [ ]:
X_train = df_train['utterance']
y_train = df_train['intent']

In [ ]:
X_test = df_test['utterance'].tolist()
y_test = df_test['intent'].tolist()

In [ ]:
print(f"Total Training samples loaded: {len(X_train)}")
print(f"Total Full Test samples loaded: {len(df_test)}")
print(f"Benchmark Test Subset ready: {len(X_test)} samples")

Total Training samples loaded: 15000
Total Full Test samples loaded: 4500
Benchmark Test Subset ready: 4500 samples


## Base line model TF-IDF+LR

In [ ]:
baseline_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

In [ ]:
print("Training the baseline model on 15,000 samples")
start_train_time = time.time()
baseline_model.fit(X_train, y_train)
train_time = time.time() - start_train_time
print(f"Training complete in {train_time:.2f} seconds.")

Training the baseline model on 15,000 samples
Training complete in 24.60 seconds.


In [ ]:
print("Running inference on the test subset...")
start_infer_time = time.time()
baseline_preds = baseline_model.predict(X_test)
baseline_infer_time = time.time() - start_infer_time

Running inference on the test subset...


In [ ]:
baseline_acc = accuracy_score(y_test, baseline_preds)
baseline_f1 = f1_score(y_test, baseline_preds, average='weighted')

In [ ]:
print(f"=== Baseline Model Results ===")
print(f"Accuracy on subset: {baseline_acc * 100:.2f}%")
print(f"f1 on subset: {baseline_f1 * 100:.2f}%")
print(f"Inference Time for {len(X_test)} samples: {baseline_infer_time:.4f} seconds")

=== Baseline Model Results ===
Accuracy on subset: 88.84%
f1 on subset: 88.76%
Inference Time for 4500 samples: 0.0823 seconds


## Zero-shot model

In [ ]:
target_intents = ['translate', 'transfer', 'timer', 'flight_status', 'weather']
df_test_subset = df_test[df_test['intent'].isin(target_intents)].groupby('intent').sample(n=20, random_state=42)

In [ ]:
X_test_subset = df_test_subset['utterance'].tolist()
y_test_subset = df_test_subset['intent'].tolist()

In [ ]:
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli",
                      device=0)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
zero_shot_preds = []
start_infer_time = time.time()

for text in X_test_subset:
    result = classifier(text, candidate_labels=target_intents)
    top_prediction = result['labels'][0]
    zero_shot_preds.append(top_prediction)

bart_infer_time = time.time() - start_infer_time

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
bart_acc = accuracy_score(y_test_subset, zero_shot_preds)
bart_f1 = f1_score(y_test_subset, zero_shot_preds, average='weighted')

In [ ]:
print(f"\n=== Zero-Shot (BART) Results ===")
print(f"Accuracy: {bart_acc * 100:.2f}%")
print(f"F1 Score: {bart_f1 * 100:.4f}%")
print(f"Inference Time: {bart_infer_time:.4f} seconds")


=== Zero-Shot (BART) Results ===
Accuracy: 89.00%
F1 Score: 88.4256%
Inference Time: 10.9924 seconds


## Held-out intent simulation

In [ ]:
# Hold out 10 intents from TF-IDF training entirely.
# TF-IDF has never seen these → collapses.
# BART zero-shot gets them cold → holds up.

In [ ]:
import random
random.seed(42)

In [ ]:
all_intents = df_train['intent'].unique().tolist()

In [ ]:
held_out_intents = random.sample(all_intents, 10)
seen_intents = [i for i in all_intents if i not in held_out_intents]

In [ ]:
print(f"Total intents: {len(all_intents)}")
print(f"Held-out intents ({len(held_out_intents)}): {held_out_intents}")
print(f"TF-IDF trains on: {len(seen_intents)} intents")

Total intents: 150
Held-out intents (10): ['rollover_401k', 'find_phone', 'report_lost_card', 'schedule_maintenance', 'uber', 'restaurant_suggestion', 'confirm_reservation', 'change_ai_name', 'maybe', 'weather']
TF-IDF trains on: 140 intents


In [ ]:
# retraining tf-idf on only 140 intents

In [ ]:
df_train_seen = df_train[df_train['intent'].isin(seen_intents)]

In [ ]:
baseline_model_b = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

In [ ]:
start = time.time()
baseline_model_b.fit(df_train_seen['utterance'], df_train_seen['intent'])
print(f"Training done in {time.time() - start:.2f}s")

Training done in 21.53s


In [ ]:
# Test set: only the 10 held-out intents, 20 samples each = 200 samples
df_heldout_test = df_test[df_test['intent'].isin(held_out_intents)]\
                    .groupby('intent').sample(n=20, random_state=42)

X_heldout = df_heldout_test['utterance'].tolist()
y_heldout = df_heldout_test['intent'].tolist()

print(f"Held-out test samples: {len(X_heldout)}")
print(f"Intents: {df_heldout_test['intent'].unique()}")

Held-out test samples: 200
Intents: ['change_ai_name' 'confirm_reservation' 'find_phone' 'maybe'
 'report_lost_card' 'restaurant_suggestion' 'rollover_401k'
 'schedule_maintenance' 'uber' 'weather']


In [ ]:
start = time.time()
tfidf_heldout_preds = baseline_model_b.predict(X_heldout)
tfidf_infer_time_b = time.time() - start

tfidf_acc_b = accuracy_score(y_heldout, tfidf_heldout_preds)
tfidf_f1_b = f1_score(y_heldout, tfidf_heldout_preds, average='weighted')

print(f"=== TF-IDF on Held-Out Intents ===")
print(f"Accuracy : {tfidf_acc_b * 100:.2f}%")
print(f"F1 Score : {tfidf_f1_b * 100:.2f}%")
print(f"Inference: {tfidf_infer_time_b:.4f}s")

=== TF-IDF on Held-Out Intents ===
Accuracy : 0.00%
F1 Score : 0.00%
Inference: 0.0176s


TF-IDF was trained on 140 intents. When you call baseline_model_b.predict(X_heldout), it predicts one of those 140 known labels — but y_heldout contains the 10 held-out labels like rollover_401k, uber, weather etc. Since none of the 10 held-out labels exist in TF-IDF's output space, every single prediction is wrong by definition. Accuracy = 0.00%.

In [ ]:
## BART Zero-Shot inference on same held-out test set
# BART zero-shot — gets the 10 held-out intents as candidate labels
# No training needed, works purely from label semantics

In [ ]:
zero_shot_preds_b = []
start = time.time()

for text in X_heldout:
    result = classifier(text, candidate_labels=held_out_intents)
    zero_shot_preds_b.append(result['labels'][0])

bart_infer_time = time.time() - start

In [ ]:
bart_acc_b = accuracy_score(y_heldout, zero_shot_preds_b)
bart_f1_b = f1_score(y_heldout, zero_shot_preds_b, average='weighted', zero_division=0)

In [ ]:
print(f"\n=== BART Zero-Shot on Held-Out Intents ===")
print(f"Accuracy : {bart_acc_b * 100:.2f}%")
print(f"F1 Score : {bart_f1_b * 100:.2f}%")
print(f"Inference: {bart_infer_time:.4f}s")


=== BART Zero-Shot on Held-Out Intents ===
Accuracy : 54.00%
F1 Score : 56.58%
Inference: 53.4169s


In [ ]:
# lets try humanize your candidate labels

In [ ]:
intent_label_map = {
    'rollover_401k'       : 'roll over 401k retirement account',
    'find_phone'          : 'find my phone',
    'report_lost_card'    : 'report a lost card',
    'schedule_maintenance': 'schedule a maintenance appointment',
    'uber'                : 'get an uber ride',
    'restaurant_suggestion': 'suggest a restaurant',
    'confirm_reservation' : 'confirm a reservation',
    'change_ai_name'      : 'change the AI assistant name',
    'maybe'               : 'uncertain or maybe response',
    'weather'             : 'ask about the weather'
}

In [ ]:
# Use human labels for BART, but map back to original for scoring
human_labels = [intent_label_map[i] for i in held_out_intents]
reverse_map = {v: k for k, v in intent_label_map.items()}

In [ ]:
zero_shot_preds_b = []
start = time.time()

for text in X_heldout:
    result = classifier(text, candidate_labels=human_labels)
    top_human_label = result['labels'][0]
    zero_shot_preds_b.append(reverse_map[top_human_label])

bart_infer_time_b = time.time() - start

In [ ]:
bart_acc_b = accuracy_score(y_heldout, zero_shot_preds_b)
bart_f1_b = f1_score(y_heldout, zero_shot_preds_b, average='weighted')

print(f"\n=== BART Zero-Shot on Held-Out Intents ===")
print(f"Accuracy : {bart_acc_b * 100:.2f}%")
print(f"F1 Score : {bart_f1_b * 100:.2f}%")
print(f"Inference: {bart_infer_time_b:.4f}s")


=== BART Zero-Shot on Held-Out Intents ===
Accuracy : 70.50%
F1 Score : 71.00%
Inference: 44.2351s


label quality directly impacts zero-shot performance — humanizing the intent names improved accuracy significantly. This is a unique challenge that supervised models like TF-IDF never face, since they learn from data regardless of what the label is called.

## Out-Of-Scope (OOS) Detection

In [ ]:
with open('/content/drive/MyDrive/DeepLearning/zero-shot dynamic routing system/data_oos_plus.json','r') as f:
  data_oos = json.load(f)

In [ ]:
oos_samples = data_oos['oos_test'][:200]
X_oos = [s[0] for s in oos_samples]
y_oos = [s[1] for s in oos_samples]

print(f"OOS test samples: {len(X_oos)}")
print(f"Sample utterances:")
for i in X_oos[:3]:
    print(f" '{i}'")

OOS test samples: 200
Sample utterances:
 'how much has the dow changed today'
 'how many prime numbers are there between 0 and 100'
 'can you tell me how to solve simple algebraic equations with one variable'


In [ ]:
# TF-IDF on OOS utterances

start = time.time()
tfidf_oos_preds = baseline_model.predict(X_oos)
tfidf_oos_proba = baseline_model.predict_proba(X_oos)
tfidf_oos_infer_time = time.time() - start

In [ ]:
tfidf_max_conf = tfidf_oos_proba.max(axis=1)

In [ ]:
tfidf_oos_correct = sum(1 for p in tfidf_oos_preds if p == 'oos')

In [ ]:
print(f"=== TF-IDF on OOS Utterances ===")
print(f"Correctly flagged as OOS : {tfidf_oos_correct} / {len(X_oos)} (0% — expected)")
print(f"Min confidence: {tfidf_max_conf.min() * 100:.1f}%")
print(f"Max confidence: {tfidf_max_conf.max() * 100:.1f}%")
print(f"Inference time: {tfidf_oos_infer_time:.4f}s")

=== TF-IDF on OOS Utterances ===
Correctly flagged as OOS : 0 / 200 (0% — expected)
Min confidence: 1.3%
Max confidence: 87.0%
Inference time: 0.0476s


In [ ]:
# BART Zero-Shot on OOS utterances

threshold_results = {}
thresholds = [0.30, 0.40, 0.50]

In [ ]:
bart_scores = []
start = time.time()

for text in X_oos:
    result = classifier(text, candidate_labels=all_intents)
    bart_scores.append(result['scores'][0])

bart_oos_infer_time = time.time() - start

In [ ]:
print(f"\nInference done in {bart_oos_infer_time:.2f}s")


Inference done in 569.48s


In [ ]:
for thresh in thresholds:
    flagged = sum(1 for s in bart_scores if s < thresh)
    detection_rate = flagged / len(bart_scores) * 100
    threshold_results[thresh] = detection_rate
    print(f"Threshold {thresh} → OOS detected: {flagged}/200 ({detection_rate:.1f}%)")

Threshold 0.3 → OOS detected: 200/200 (100.0%)
Threshold 0.4 → OOS detected: 200/200 (100.0%)
Threshold 0.5 → OOS detected: 200/200 (100.0%)


In [ ]:
best_thresh = max(threshold_results, key=threshold_results.get)
best_rate   = threshold_results[best_thresh]

oos_summary = pd.DataFrame({
    'Model'              : ['TF-IDF + LR', f'BART Zero-Shot (thresh={best_thresh})'],
    'OOS Detection Rate' : [f'0.0%', f'{best_rate:.1f}%'],
    'Avg Confidence'     : [f'{tfidf_max_conf.mean()*100:.1f}%',
                            f'{sum(bart_scores)/len(bart_scores)*100:.1f}%'],
    'Behaviour'          : ['Always predicts a known intent',
                            'Flags low-confidence as OOS']
})

print("=== Scenario C: OOS Detection Summary ===")
print(oos_summary.to_string(index=False))
print()
print("Key insight: TF-IDF is architecturally incapable of detecting OOS.")
print(f"BART detects {best_rate:.0f}% of OOS utterances at threshold={best_thresh} — no retraining needed.")

=== Scenario C: OOS Detection Summary ===
                      Model OOS Detection Rate Avg Confidence                      Behaviour
                TF-IDF + LR               0.0%           8.7% Always predicts a known intent
BART Zero-Shot (thresh=0.3)             100.0%           5.8%    Flags low-confidence as OOS

Key insight: TF-IDF is architecturally incapable of detecting OOS.
BART detects 100% of OOS utterances at threshold=0.3 — no retraining needed.
